# Compare Text-to-Cypher inference time

Notebook so sánh thời gian inference của bốn cấu hình:

1. Base `Qwen3-0.6B`
2. SFT `Qwen3-0.6B`
3. Teacher LoRA `Qwen3-4B-Instruct-2507`
4. CypherKD `Qwen3-0.6B` LoRA

Mỗi model được load và benchmark tuần tự rồi giải phóng khỏi GPU. Thời gian load/merge LoRA được báo riêng và **không tính vào inference latency**.

## Kaggle setup

Bật Internet và GPU trong Kaggle. Nếu clone repo vào `/kaggle/working`:

```python
!apt-get update -qq
!apt-get install -y -qq libaio-dev
%pip install -q transformers==4.57.3 peft==0.18.1 accelerate huggingface-hub deepspeed
!git clone -q https://github.com/fisherman611/CypherKD-draft.git /kaggle/working/CypherKD-draft
```

Các Qwen base models là public. Chỉ cần `HF_TOKEN` nếu checkpoint adapter là private hoặc gặp giới hạn tải.

In [ ]:
from pathlib import Path
import gc
import json
import os
import statistics
import sys
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

root_candidates = [
    Path.cwd(),
    Path.cwd() / "CypherKD-draft",
    Path.cwd() / "CypherKD",
    Path("/kaggle/working/CypherKD-draft"),
    Path.cwd().parent,
]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "benchmarks").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Không tìm thấy repo CypherKD. Hãy clone/upload repo trước.")
sys.path.insert(0, str(PROJECT_ROOT))

if not torch.cuda.is_available():
    raise RuntimeError("Benchmark này cần GPU CUDA để kết quả có ý nghĩa.")

DEVICE = "cuda:0"
# scripts dùng configs/deepspeed/ds_config_bf16.json. Fallback FP16 chỉ dành
# cho GPU Kaggle không hỗ trợ BF16 (ví dụ T4), và sẽ được báo rõ trong kết quả.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Project root: {PROJECT_ROOT}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"Dtype       : {DTYPE}")
if DTYPE != torch.bfloat16:
    print("WARNING: scripts dùng BF16, nhưng GPU này không hỗ trợ; benchmark fallback sang FP16.")

## 1. Điền checkpoint

Checkpoint có thể là đường dẫn local/Kaggle Dataset hoặc Hugging Face repo ID. Các cấu hình còn placeholder sẽ được đánh dấu `SKIPPED`; base model vẫn chạy bình thường.

Teacher và CypherKD dùng đúng base model/LoRA assumptions từ scripts (`r=32`, `alpha=64`, `dropout=0.1`; các giá trị này đã nằm trong `adapter_config.json`). Repo hiện không có script SFT riêng cho student 0.6B, nên dòng SFT dùng cùng base/inference config 0.6B và chờ adapter placeholder.

In [ ]:
BASE_06B = "Qwen/Qwen3-0.6B"
BASE_TEACHER_4B = "Qwen/Qwen3-4B-Instruct-2507"

# TODO: thay ba placeholder này bằng checkpoint thật.
SFT_06B_ADAPTER = "<PATH_OR_HF_REPO_TO_SFT_QWEN3_06B_ADAPTER>"
TEACHER_4B_ADAPTER = "<PATH_OR_HF_REPO_TO_TEACHER_4B_LORA_ADAPTER>"
CYPHERKD_06B_ADAPTER = "<PATH_OR_HF_REPO_TO_CYPHERKD_06B_ADAPTER>"

MODEL_SPECS = [
    {
        "name": "Base Qwen3-0.6B",
        "base_model": BASE_06B,
        "adapter": None,
        "source_script": "scripts/process_cypherbench_qwen3_0.6B.sh",
    },
    {
        "name": "SFT Qwen3-0.6B",
        "base_model": BASE_06B,
        "adapter": SFT_06B_ADAPTER,
        "source_script": "No SFT 0.6B script in scripts/; shared 0.6B inference config",
    },
    {
        "name": "Teacher Qwen3-4B LoRA",
        "base_model": BASE_TEACHER_4B,
        "adapter": TEACHER_4B_ADAPTER,
        "source_script": "scripts/teacher_lora/lora_qwen3_4B.sh",
    },
    {
        "name": "CypherKD Qwen3-0.6B LoRA",
        "base_model": BASE_06B,
        "adapter": CYPHERKD_06B_ADAPTER,
        "source_script": "scripts/cypherkd/cypherkd_qwen3_0.6B_4B.sh",
    },
]

pd.DataFrame(MODEL_SPECS)

## 2. Cấu hình benchmark

- Cùng ba samples từ `test.jsonl` cho cả bốn model.
- Sampling đúng scripts: `do_sample=True`, `top_p=0.95`, `top_k=0`, `temperature=0.5`, `num_beams=1`.
- `seed=42`, `max_length=892`, `max_prompt_length=797` như teacher/CypherKD scripts.
- Warm-up riêng cho từng model.
- `torch.cuda.synchronize()` trước và sau mỗi phép đo.
- Báo cáo median/P95 và tokens/s vì các model có thể sinh số token khác nhau.

In [ ]:
# Ba test samples có prompt vừa phải, không có Unicode escape và Cypher đa dạng.
SAMPLE_INDICES = [2167, 907, 1587]
WARMUP_RUNS = 2
MEASURED_RUNS = 5

# Đồng bộ scripts/teacher_lora và scripts/cypherkd.
SEED = 42
MAX_SEQUENCE_LENGTH = 892
MAX_PROMPT_LENGTH = 797
DO_SAMPLE = True
TOP_K = 0
TOP_P = 0.95
TEMPERATURE = 0.5
NUM_BEAMS = 1
OUTPUT_DIR = PROJECT_ROOT / "results/inference_time_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert WARMUP_RUNS >= 0
assert MEASURED_RUNS > 0
assert MAX_SEQUENCE_LENGTH > MAX_PROMPT_LENGTH

print({
    "seed": SEED,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "do_sample": DO_SAMPLE,
    "top_k": TOP_K,
    "top_p": TOP_P,
    "temperature": TEMPERATURE,
    "num_beams": NUM_BEAMS,
})

## 3. Chuẩn bị ba Text-to-Cypher prompts từ test.jsonl

In [ ]:
test_path = PROJECT_ROOT / "benchmarks/Cypherbench/test.jsonl"
selected_rows = {}
with test_path.open("r", encoding="utf-8") as handle:
    for row_index, line in enumerate(handle):
        if row_index in SAMPLE_INDICES:
            selected_rows[row_index] = json.loads(line)
        if len(selected_rows) == len(SAMPLE_INDICES):
            break

missing_indices = [index for index in SAMPLE_INDICES if index not in selected_rows]
if missing_indices:
    raise IndexError(f"Không tìm thấy test.jsonl indices: {missing_indices}")

benchmark_samples = []
for sample_index in SAMPLE_INDICES:
    row = selected_rows[sample_index]
    user_prompt = row["user_prompt"]
    question = user_prompt.split("QUESTION:\n", 1)[-1].split("\n\nSCHEMA:\n", 1)[0]
    gold_cypher = json.loads(row["response"])["cypher"]
    benchmark_samples.append({
        "sample_index": sample_index,
        "question": question,
        "gold_cypher": gold_cypher,
        "messages": [
            {"role": "system", "content": row["system_prompt"]},
            {"role": "user", "content": user_prompt},
        ],
    })

for item in benchmark_samples:
    print(f"[{item['sample_index']}] {item['question']}")
    print(f"Gold: {item['gold_cypher']}\n")

## 4. Benchmark helpers

In [ ]:
def is_placeholder(value):
    return isinstance(value, str) and value.startswith("<") and value.endswith(">")

def percentile(values, q):
    ordered = sorted(float(value) for value in values)
    position = (len(ordered) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1 - fraction) + ordered[upper] * fraction

def apply_chat_template(tokenizer, chat_messages):
    try:
        return tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def load_model_and_tokenizer(spec):
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        spec["base_model"], padding_side="left", trust_remote_code=True
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        spec["base_model"],
        torch_dtype=DTYPE,
        device_map={"": DEVICE},
        trust_remote_code=True,
    )
    if spec["adapter"] is not None:
        model = PeftModel.from_pretrained(model, spec["adapter"])
        model = model.merge_and_unload()
    model.eval()
    torch.cuda.synchronize()
    return tokenizer, model, time.perf_counter() - started

def generate_once(tokenizer, model, model_inputs):
    # Reset seed để repeated runs dùng cùng sampling stream và giảm timing noise.
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    prompt_tokens = int(model_inputs["input_ids"].shape[1])
    max_new_tokens = MAX_SEQUENCE_LENGTH - prompt_tokens
    if max_new_tokens <= 0:
        raise ValueError(
            f"Prompt có {prompt_tokens} tokens, vượt max sequence length "
            f"{MAX_SEQUENCE_LENGTH}."
        )
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            top_k=TOP_K,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=list(dict.fromkeys([tokenizer.eos_token_id, 151643])),
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated_ids = output[0, prompt_tokens:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response, int(generated_ids.numel()), elapsed

def release_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

## 5. Chạy tuần tự bốn cấu hình

Không chạy tác vụ GPU khác trong lúc benchmark. Với checkpoint placeholder, cấu hình tương ứng sẽ được skip thay vì làm notebook lỗi.

In [ ]:
results = []

for spec in MODEL_SPECS:
    print("=" * 88)
    print(spec["name"])
    if is_placeholder(spec["adapter"]):
        print(f"SKIPPED: hãy thay placeholder {spec['adapter']}")
        results.append({
            "model": spec["name"],
            "status": "SKIPPED",
            "base_model": spec["base_model"],
            "adapter": spec["adapter"],
            "source_script": spec["source_script"],
        })
        continue

    tokenizer = model = model_inputs = None
    try:
        tokenizer, model, load_seconds = load_model_and_tokenizer(spec)
        print(f"Load + merge: {load_seconds:.3f} s")
        warmup_prompt = apply_chat_template(tokenizer, benchmark_samples[0]["messages"])
        model_inputs = tokenizer(
            warmup_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
        ).to(DEVICE)
        for warmup_idx in range(WARMUP_RUNS):
            _, _, warmup_time = generate_once(tokenizer, model, model_inputs)
            print(f"Warm-up {warmup_idx + 1}/{WARMUP_RUNS}: {warmup_time:.3f} s")

        torch.cuda.reset_peak_memory_stats()
        all_latencies = []
        all_output_lengths = []
        all_prompt_lengths = []
        sample_results = []

        for sample_item in benchmark_samples:
            prompt_text = apply_chat_template(tokenizer, sample_item["messages"])
            model_inputs = tokenizer(
                prompt_text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_PROMPT_LENGTH,
            ).to(DEVICE)
            prompt_tokens = int(model_inputs["input_ids"].shape[1])
            sample_latencies = []
            sample_output_lengths = []
            response = ""
            print(f"\nSample {sample_item['sample_index']} ({prompt_tokens} prompt tokens)")
            for run_idx in range(MEASURED_RUNS):
                response, output_tokens, elapsed = generate_once(tokenizer, model, model_inputs)
                sample_latencies.append(elapsed)
                sample_output_lengths.append(output_tokens)
                print(
                    f"  Run {run_idx + 1}/{MEASURED_RUNS}: "
                    f"{elapsed:.3f} s, {output_tokens} tokens"
                )

            sample_result = {
                "sample_index": sample_item["sample_index"],
                "question": sample_item["question"],
                "gold_cypher": sample_item["gold_cypher"],
                "prompt_tokens": prompt_tokens,
                "average_latency_seconds": statistics.mean(sample_latencies),
                "median_latency_seconds": statistics.median(sample_latencies),
                "p95_latency_seconds": percentile(sample_latencies, 0.95),
                "average_output_tokens": statistics.mean(sample_output_lengths),
                "latencies_seconds": sample_latencies,
                "output_token_counts": sample_output_lengths,
                "response": response,
            }
            sample_results.append(sample_result)
            all_latencies.extend(sample_latencies)
            all_output_lengths.extend(sample_output_lengths)
            all_prompt_lengths.append(prompt_tokens)
            print(f"  Sample average: {sample_result['average_latency_seconds']:.3f} s")

        average_seconds = statistics.mean(all_latencies)
        median_seconds = statistics.median(all_latencies)
        average_output_tokens = statistics.mean(all_output_lengths)
        result = {
            "model": spec["name"],
            "status": "OK",
            "base_model": spec["base_model"],
            "adapter": spec["adapter"],
            "source_script": spec["source_script"],
            "dtype": str(DTYPE),
            "generation_config": {
                "seed": SEED,
                "max_sequence_length": MAX_SEQUENCE_LENGTH,
                "max_prompt_length": MAX_PROMPT_LENGTH,
                "do_sample": DO_SAMPLE,
                "top_k": TOP_K,
                "top_p": TOP_P,
                "temperature": TEMPERATURE,
                "num_beams": NUM_BEAMS,
            },
            "load_seconds": load_seconds,
            "sample_count": len(benchmark_samples),
            "average_prompt_tokens": statistics.mean(all_prompt_lengths),
            "average_output_tokens": average_output_tokens,
            "average_latency_seconds": average_seconds,
            "median_latency_seconds": median_seconds,
            "p95_latency_seconds": percentile(all_latencies, 0.95),
            "tokens_per_second": sum(all_output_lengths) / sum(all_latencies),
            "peak_vram_gib": torch.cuda.max_memory_allocated() / (1024 ** 3),
            "sample_results": sample_results,
        }
        results.append(result)
        print(f"\nAverage over all samples/runs: {average_seconds:.3f} s")
        print(f"Median : {median_seconds:.3f} s")
        print(f"P95   : {result['p95_latency_seconds']:.3f} s")
        print(f"Speed : {result['tokens_per_second']:.2f} tokens/s")
        print(f"VRAM  : {result['peak_vram_gib']:.2f} GiB")
    except Exception as exc:
        print(f"FAILED: {type(exc).__name__}: {exc}")
        results.append({
            "model": spec["name"],
            "status": "FAILED",
            "base_model": spec["base_model"],
            "adapter": spec["adapter"],
            "source_script": spec["source_script"],
            "error": f"{type(exc).__name__}: {exc}",
        })
    finally:
        model_inputs = None
        model = None
        tokenizer = None
        release_gpu()


## 6. Bảng so sánh và lưu kết quả

`average_latency_seconds` là trung bình cộng của tất cả `model.generate()` trên 3 samples × `MEASURED_RUNS`, sau warm-up. Giá trị này không gồm load model, tokenizer, merge LoRA hay đọc dữ liệu.

In [ ]:
summary_columns = [
    "model",
    "status",
    "source_script",
    "dtype",
    "load_seconds",
    "sample_count",
    "average_prompt_tokens",
    "average_output_tokens",
    "average_latency_seconds",
    "median_latency_seconds",
    "p95_latency_seconds",
    "tokens_per_second",
    "peak_vram_gib",
]
summary = pd.DataFrame(results).reindex(columns=summary_columns)
display(summary)

json_path = OUTPUT_DIR / "inference_time_results.json"
csv_path = OUTPUT_DIR / "inference_time_summary.csv"
with json_path.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, ensure_ascii=False, indent=2, default=str)
summary.to_csv(csv_path, index=False)

print(f"JSON: {json_path}")
print(f"CSV : {csv_path}")

## Cách diễn giải

- Dùng **median latency** thay vì một lần chạy đơn lẻ.
- So thêm **tokens/s** vì output có thể kết thúc ở độ dài khác nhau.
- Không dùng `load_seconds` để kết luận tốc độ inference; đó là startup cost.
- `average_latency_seconds` trả lời trực tiếp yêu cầu thời gian trung bình trên ba samples; median/P95 giúp phát hiện kết quả bị chi phối bởi outlier.
- Nếu cần so sánh tuyệt đối với output dài cố định, chọn các samples mà bốn model sinh output có độ dài gần nhau; nếu không, ưu tiên thêm chỉ số tokens/s.